In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt


In [2]:

met_labels = pd.read_csv(r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\analysis\20240911_seaad\MET_typing\config\met_cell_assignments.csv', index_col='specimen_id')
met_labels.met_type = met_labels.met_type.apply(lambda x: f'MET_{x}')

met_labels_four= pd.read_csv(r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\analysis\20240911_seaad\MET_typing\config_tree_top_four\met_cell_assignments.csv', index_col='specimen_id')
met_labels_four.met_type = met_labels_four.met_type.apply(lambda x: f'4MET_{x}')
met_labels_four = met_labels_four.rename(columns={'met_type' :'met_type_four'})


out_dir = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\analysis\20240911_seaad\MET_typing\config_tree_top_four'

river_type = 'met-met4' 


In [3]:
# combine the data 
data = met_labels.merge(met_labels_four, how='inner', left_index=True, right_index=True)
data

,met_type,met_type_four
specimen_id,,
541549258,MET_0,4MET_0
541557114,MET_11,4MET_11
592532014,MET_5,4MET_5
595572609,MET_13,4MET_14
595583062,MET_6,4MET_6
...,...,...
1187944209,MET_10,4MET_10
1187963672,MET_10,4MET_10
1157661104,MET_18,4MET_17


In [4]:
# met cluster order 
met_ordered_cluster = data['met_type'].unique()
met_ordered_cluster = sorted(met_ordered_cluster, key=lambda x: int(x.split('_')[1]))
met_ordered_cluster = list(met_ordered_cluster)

# met cluster order 
met_ordered_cluster_four = data['met_type_four'].unique()
met_ordered_cluster_four = sorted(met_ordered_cluster_four, key=lambda x: int(x.split('_')[1]))
met_ordered_cluster_four = list(met_ordered_cluster_four)


In [5]:
# Choose a colormap (e.g., 'viridis', 'plasma', 'coolwarm', etc.)
colormap = plt.get_cmap("viridis")

# Sample n colors evenly from the colormap
colors = [colormap(i / (len(met_ordered_cluster) - 1)) for i in range(len(met_ordered_cluster))]  # Get n colors

# Convert RGBA to hex and create a dictionary mapping each string to a color
met_color_dict = {cluster: '#' + ''.join(f'{int(c * 255):02x}' for c in colors[i][:3]) for i, cluster in enumerate(met_ordered_cluster)}


In [6]:
source_target_pairs = {
    'met-met4':['met_type', 'met_type_four', met_color_dict, met_ordered_cluster, met_ordered_cluster_four],   
}

[source_col, target_col, node_colors, source_nodes, target_nodes ] = source_target_pairs[river_type]

In [7]:
# Create nodes
nodes = source_nodes + target_nodes

# Create links
links = []
for i in range(len(data)):
    source_idx = nodes.index(data[source_col].iloc[i])
    target_idx = nodes.index(data[target_col].iloc[i])
    links.append({source_col: source_idx, target_col: target_idx, 'value': 1, 'source_color': node_colors[data[source_col].iloc[i]]})

# Define Sankey plot
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="white", width=0.5),
        label=nodes,
        # color=[node_colors[node] for node in nodes]  # Assigning colors based on the node_colors dictionary
        color = [node_colors.get(node, '#808080') for node in nodes]


    ),
    link=dict(
        source=[link[source_col] for link in links],
        target=[link[target_col] for link in links],
        value=[link['value'] for link in links],
        color=[link['source_color'] for link in links]  #NOTE uncomment to plot links as source node color

    ),
    arrangement="snap"
)])


# Update layout
fig.update_layout(title_text="{} to {}".format(source_col, target_col), font_size=10,width=300, height=1000)

# Save figure 
fig.write_html(os.path.join(out_dir, "{} to {}.html".format(source_col, target_col)))
fig.write_image(os.path.join(out_dir, "{} to {}.pdf".format(source_col, target_col)), format="pdf")

# Show plot
fig.show()
